# Lecture12

In this assignment you will implement a 3D convolutional neural network (CNN) architecture.
The CNN should have the following layers (listed in order):


- **Input:** 3D tensor with size 128 x 256 x 256
  
- **Layer 0:** Max Pool 3D, downsampling **x4** in each spatial dimension

- **Layer 1:** Convolution 3D (number of kernels = 8, kernel size: 3, stide: 1, zero padding)
  
- **Layer 2:** ReLU

- **Layer 3:** Max Pool 3D (downsampling **x2** in each spatial dimension)

- **Layer 4:** Convolution 3D (number of kernels = 16, kernel size: 5, stide: 1, zero padding)

- **Layer 5:** ReLU

- **Layer 6:** Max Pool 3D (downsampling **x4** in each spatial dimension)

- **Layer 7:** Flatten to vector 

- **Layer 8:** Fully-connected, hidden units: 512

- **Layer 9:** ReLU

- **Layer 10:** Fully-connected, hidden units: 2

In [2]:
import torch
import torch.nn as nn
import SimpleITK as sitk
import numpy as np
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

In [3]:
if torch.backends.mps.is_available():
    print("MPS is available.")
    device = torch.device("mps")
elif torch.cuda.is_available():
    print("CUDA is available.")
    device = torch.device("cuda")
else:
    print("Using CPU")
    device = torch.device("cpu")


engr_dir = "/opt/nfsopt/DLMI"
idas_dir = os.path.join(os.path.expanduser('~'), "classdata")

if os.path.isdir(engr_dir):
    data_dir = engr_dir
elif os.path.isdir(idas_dir):  
    data_dir = idas_dir
else:
    print("Data directory not found")

Using CPU


/home/iabney/.local/lib/python3.11/site-packages/torch/cuda/__init__.py:228: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## Part 1

Before implementing the CNN model in PyTorch, first calculate the output size of each layer and the number of learnable parameters associated with each layer. **This part should be completed by hand and upload as a separate PDF. Show all work.** Below enter the total number of learnable parameters calculated.

## Spacial padding should equal 0 padding

## Part 2

Implement the above 3D CNN by creating a Python class which inherits from the `torch.nn.Module` class. Your class will need to implement the `__init__` and `forward` methods . The `__init__` method should just create instances of all layers, and the `forward` method should perform the forward pass by composing the layers together and returning the output. Refer to Lecture09 slides for a refresher on how to create a custom model using `torch.nn.Module`. 

In [17]:
class CNN(nn.Module):
    ######################################
    #######         TODO           #######
    ######################################
    
    
    def __init__(self):
        super(CNN, self).__init__()
        
        # Layer 0: Max Pool 3D (downsampling x4 in spatial dimensions)
        self.pool0 = nn.MaxPool3d(kernel_size=(1, 4, 4), stride=(1, 4, 4))
        
        # Layer 1: Convolution 3D (8 kernels, kernel size: 3, stride: 1, same padding)
        self.conv1 = nn.Conv3d(in_channels=1, out_channels=8, kernel_size=3, stride=1, padding=1)
        
        # Layer 2: ReLU
        self.relu2 = nn.ReLU()
        
        # Layer 3: Max Pool 3D (downsampling x2 in spatial dimensions)
        self.pool3 = nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))
        
        # Layer 4: Convolution 3D (16 kernels, kernel size: 5, stride: 1, same padding)
        self.conv4 = nn.Conv3d(in_channels=8, out_channels=16, kernel_size=5, stride=1, padding=2)
        
        # Layer 5: ReLU
        self.relu5 = nn.ReLU()
        
        # Layer 6: Max Pool 3D (downsampling x4 in spatial dimensions)
        self.pool6 = nn.MaxPool3d(kernel_size=(1, 4, 4), stride=(1, 4, 4))
        
        # Layer 7: Flatten
        self.flatten7 = nn.Flatten()
        
        # Layer 8: Fully-connected (Hidden units: 512)
        # Input size calculation: 16 channels * 128 depth * 8 height * 8 width = 131,072
        self.fc8 = nn.Linear(in_features=16 * 128 * 8 * 8, out_features=512)
        
        # Layer 9: ReLU
        self.relu9 = nn.ReLU()
        
        # Layer 10: Fully-connected (Hidden units: 2)
        self.fc10 = nn.Linear(in_features=512, out_features=2)

    def forward(self, x):
        x = self.pool0(x)
        x = self.conv1(x)
        x = self.relu2(x)
        x = self.pool3(x)
        x = self.conv4(x)
        x = self.relu5(x)
        x = self.pool6(x)
        x = self.flatten7(x)
        x = self.fc8(x)
        x = self.relu9(x)
        x = self.fc10(x)
        return x


# confirm model 
if __name__ == "__main__":
    # Batch size = 1, Channels = 1, Depth = 128, Height = 256, Width = 256
    model = CNN()
    dummy_input = torch.randn(1, 1, 128, 256, 256)
    output = model(dummy_input)
    print(f"Input shape:  {dummy_input.shape}")
    print(f"Output shape: {output.shape}")

 

Input shape:  torch.Size([1, 1, 128, 256, 256])
Output shape: torch.Size([1, 2])


Read an example MRI image to use for testing the model:

In [5]:
fn = os.path.join(data_dir, "lecture12", "mri.nii.gz")
im = sitk.ReadImage(fn)
print(f'Image size: {im.GetSize()}')

Image size: (256, 256, 128)


Convert the SimpleITK image to a NumPy array and then convert it to a PyTorch tensor.

In [26]:
######################################
#######         TODO           #######
######################################

#convert to np array
imnp = sitk.GetArrayFromImage(im)

#convert np array to tensor
tensor = torch.from_numpy(imnp)

#view tensor
tensor.shape



torch.Size([128, 256, 256])

The 3D layers in Pytorch expect the input tensors to be 5D where the dimensions correspond to 

(batch, channels, size_k, size_j, size_i)


In this example, both the batch and channel size will be 1 for the input, however, a placeholder/dummy dimension still needs to be created or Pytorch will throw an error. 

Calling `unsqueeze(dim)` on a tensor will create a new dimension (with size 1) at the specified `dim` position. Convert the above 3D tensor to the apporpirate 5D tensor. Additionally, convert the tensor data type to `float`. Print the resulting shape and data type to confirm the desired properties.

In [29]:
######################################
#######         TODO           #######
######################################
tensor_5d = tensor.unsqueeze(0).unsqueeze(0).float()

tensor_5d.shape


Create an instance of the CNN class and peform a forward pass using the prepared tensor as input. To run on the GPU, you will need to put both the model and input tensor on the GPU before calling the forward pass using `to(device)`

In [32]:
######################################
#######         TODO           #######
######################################

# Instantiate the model
model = CNN()

#send to device
tensor_5d.to(device)
model.to(device)

#Set model to evaluation mode (good practice for inference)
model.eval()

# Perform the forward pass (without computing gradients)
with torch.no_grad():
    output = model(tensor_5d)

print(f"Output tensor:\n{output}")


Output tensor:
tensor([[ 3.0060, -3.7240]])


Print the total number of learnable parameters:

In [31]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {num_params}')

Total params: 67126642


Save a PDF of the document to upload to ICON and then add/commit/push to your git repository.